# Ensemble results across machines

Explore the published outputs of the [full subset experiment](04_ensemble_combinations.ipynb). These results describe one initialization seed on a previously inspected test split.

In [1]:
from pathlib import Path
import os
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "notebooks/code").is_dir() and (p / "results/fits.json").is_file())
os.chdir(root)
import csv
import json
from IPython.display import Markdown, display

published = root / "docs/results/ensemble_subsets_seed42"
def read_rows(name):
    with (published / name).open() as stream:
        return list(csv.DictReader(stream))

rows = read_rows("all_combinations.csv")
machine_rows = read_rows("by_machine.csv")
check = json.loads((published / "split_check.json").read_text())
best = max(rows, key=lambda r: float(r["pauc"]))
original = next(r for r in rows if r["size"] == "5" and r["includes_xgboost"] == "False")
trio = max((r for r in rows if r["size"] == "3"), key=lambda r: float(r["pauc"]))
assert len(rows) == 63 and len(machine_rows) == 252

## Fault recall and false alarms

Compare the original five-CNN ensemble with the strongest three- and four-member combinations by partial AUC.

In [2]:
def show_table(items, label):
    lines = ["| Configuration | AUC (%) | pAUC (%) | Recall (%) | FPR (%) | Faults caught | False alarms |",
             "|---|---:|---:|---:|---:|---:|---:|"]
    for row in items:
        rates = [f"{100 * float(row[k]):.2f}" for k in ("auc", "pauc", "recall", "fpr")]
        lines.append("| " + " | ".join([row[label], *rates, str(row["tp"]), str(row["fp"])]) + " |")
    display(Markdown("\n".join(lines)))
show_table([original, trio, best], "members")

| Configuration | AUC (%) | pAUC (%) | Recall (%) | FPR (%) | Faults caught | False alarms |
|---|---:|---:|---:|---:|---:|---:|
| MobileNetV3-Small + ResNet-18 + EfficientNet-B0 + Compact CNN + Wide CNN | 93.62 | 88.79 | 76.69 | 8.17 | 7592 | 720 |
| ResNet-18 + EfficientNet-B0 + XGBoost | 95.58 | 90.08 | 80.66 | 7.51 | 7985 | 662 |
| ResNet-18 + EfficientNet-B0 + Wide CNN + XGBoost | 95.66 | 90.55 | 80.70 | 7.42 | 7989 | 654 |

## Results for the leading combination by machine

In [3]:
show_table([r for r in machine_rows if r["members"] == best["members"]], "machine")

| Configuration | AUC (%) | pAUC (%) | Recall (%) | FPR (%) | Faults caught | False alarms |
|---|---:|---:|---:|---:|---:|---:|
| fan | 90.34 | 83.32 | 67.95 | 7.08 | 3007 | 173 |
| pump | 93.92 | 85.90 | 69.66 | 7.48 | 953 | 168 |
| slider | 98.84 | 95.13 | 97.60 | 7.52 | 2606 | 144 |
| valve | 99.55 | 97.86 | 99.03 | 7.64 | 1423 | 169 |

## Check on a separate group of source recordings

The selection half contained 9,276 clips and the check half contained 9,438. Both come from the original test set. The comparison measures stability within that set.

In [4]:
selected = check["selected_by_first_half_pauc"]["members"]
print("Selected on first half:", selected)
show_table([dict(members=k, **v) for k, v in sorted(check["second_half_results"].items())], "members")

Selected on first half: ResNet-18 + EfficientNet-B0 + Wide CNN + XGBoost


| Configuration | AUC (%) | pAUC (%) | Recall (%) | FPR (%) | Faults caught | False alarms |
|---|---:|---:|---:|---:|---:|---:|
| MobileNetV3-Small + ResNet-18 + EfficientNet-B0 + Compact CNN + Wide CNN | 93.43 | 88.48 | 76.83 | 8.07 | 3854 | 357 |
| MobileNetV3-Small + ResNet-18 + EfficientNet-B0 + Compact CNN + Wide CNN + XGBoost | 94.81 | 89.55 | 80.46 | 8.34 | 4036 | 369 |
| ResNet-18 + EfficientNet-B0 + Wide CNN + XGBoost | 95.65 | 90.42 | 80.70 | 7.44 | 4048 | 329 |